In [1]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()
llm = ChatGroq(model="openai/gpt-oss-120b", api_key=os.getenv("GROQ_API_KEY"), temperature=0.4)

c:\Users\harih\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from pydantic import BaseModel
from typing import Literal
from langchain_core.prompts import ChatPromptTemplate

class IntentClassifier(BaseModel):
    intent: Literal["notes", "tasks", "research", "knowledge", "chat"]
    reasoning: str

classifier_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an intent classifier. Always classify the user's message into exactly one of these intents:
- notes: saving, listing, searching, updating, or deleting notes
- tasks: adding, listing, completing, or deleting tasks
- research: searching the web or Wikipedia for current information
- knowledge: querying uploaded documents
- chat: general conversation, greetings, or anything else

You MUST always return a structured classification. Never respond in plain text."""),
    ("human", "{input}")
])

classifier = classifier_prompt | llm.with_structured_output(IntentClassifier)

In [5]:
test_inputs = [
    "Save a note about my meeting with John",
    "What are my pending tasks?",
    "Search the web for latest Python news",
    "What does my uploaded document say about onboarding?",
    "How are you today?"
]

for q in test_inputs:
    result = classifier.invoke(q)
    print(f"Query: {q}")
    print(f"Intent: {result.intent} | Reason: {result.reasoning}")
    print()

Query: Save a note about my meeting with John
Intent: notes | Reason: The user explicitly wants to save a note about a meeting, which falls under the 'notes' intent for saving notes.

Query: What are my pending tasks?
Intent: tasks | Reason: The user is asking to retrieve their pending tasks, which falls under the 'tasks' intent for listing tasks.

Query: Search the web for latest Python news
Intent: research | Reason: The user explicitly asks to search the web for the latest Python news, which falls under the 'research' intent.

Query: What does my uploaded document say about onboarding?
Intent: knowledge | Reason: The user is asking for information contained in an uploaded document, which falls under the 'knowledge' intent for querying uploaded documents.

Query: How are you today?
Intent: chat | Reason: The user is asking a greeting question about the assistant's state, which falls under general conversation.



In [6]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import WikipediaQueryRun
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.tools import tool
from datetime import date
import sqlite3
from datetime import datetime, timezone

today_str = date.today().strftime("%B %d, %Y")

print("Imports done")

Imports done


C:\Users\harih\AppData\Local\Temp\ipykernel_6700\3440540292.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


In [7]:
# --- Notes tools ---
notes_conn = sqlite3.connect("notes.db", check_same_thread=False)
notes_conn.row_factory = sqlite3.Row

tasks_conn = sqlite3.connect("tasks.db", check_same_thread=False)
tasks_conn.row_factory = sqlite3.Row

def now():
    return datetime.now(timezone.utc).isoformat()

@tool
def add_note(title: str, content: str, tags: str = "") -> str:
    """Save a new note with a title, content, and optional comma-separated tags."""
    cur = notes_conn.execute(
        "INSERT INTO notes (title, content, tags, created_at) VALUES (?, ?, ?, ?)",
        (title, content, tags, now())
    )
    notes_conn.commit()
    return f"Note saved with id {cur.lastrowid}: '{title}'."

@tool
def list_notes(limit: int = 20) -> str:
    """List the most recent notes."""
    rows = notes_conn.execute(
        "SELECT id, title, tags, created_at FROM notes ORDER BY created_at DESC LIMIT ?",
        (limit,)
    ).fetchall()
    if not rows:
        return "No notes found."
    return "\n".join(f"[{r['id']}] {r['title']} (tags: {r['tags']})" for r in rows)

@tool
def search_notes(query: str) -> str:
    """Search notes by keyword. Matches singular/plural word forms."""
    words = [w.strip().lower() for w in query.split() if len(w.strip()) > 2]
    if not words:
        return "Please provide a more specific search term."
    variants = set()
    for w in words:
        variants.add(w)
        if w.endswith("s"):
            variants.add(w[:-1])
        else:
            variants.add(w + "s")
    conditions = " OR ".join(["(LOWER(title) LIKE ? OR LOWER(content) LIKE ? OR LOWER(tags) LIKE ?)"] * len(variants))
    params = []
    for v in variants:
        like = f"%{v}%"
        params.extend([like, like, like])
    rows = notes_conn.execute(
        f"SELECT id, title, content, tags FROM notes WHERE {conditions}", params
    ).fetchall()
    if not rows:
        return f"No notes matched '{query}'."
    return "\n\n".join(f"[{r['id']}] {r['title']}\n{r['content']}" for r in rows)

# --- Tasks tools ---
@tool
def add_task(description: str, due_date: str = "") -> str:
    """Add a new task with an optional due date (YYYY-MM-DD)."""
    cur = tasks_conn.execute(
        "INSERT INTO tasks (description, due_date, created_at) VALUES (?, ?, ?)",
        (description, due_date, now())
    )
    tasks_conn.commit()
    return f"Task added with id {cur.lastrowid}: '{description}'."

@tool
def list_pending_tasks() -> str:
    """List all pending tasks."""
    rows = tasks_conn.execute(
        "SELECT id, description, due_date FROM tasks WHERE status = 'pending' ORDER BY due_date"
    ).fetchall()
    if not rows:
        return "No pending tasks."
    return "\n".join(f"[{r['id']}] {r['description']} (due: {r['due_date'] or 'n/a'})" for r in rows)

@tool
def complete_task(task_id: int) -> str:
    """Mark a task as complete by id."""
    tasks_conn.execute(
        "UPDATE tasks SET status = 'complete', completed_at = ? WHERE id = ?", (now(), task_id)
    )
    tasks_conn.commit()
    return f"Task {task_id} marked complete."

print("Tools ready")

Tools ready


In [9]:
# Research agent
search_tool = DuckDuckGoSearchRun(description="Search the web for current news and information.")
wiki_api = WikipediaAPIWrapper()
wikipedia_tool = WikipediaQueryRun(api_wrapper=wiki_api)
research_toolkit = [search_tool, wikipedia_tool]

research_agent = create_agent(
    llm,
    tools=research_toolkit,
    system_prompt=f"""You are a research assistant. Today's date is {today_str}.
You have two tools: duckduckgo_search for current news, wikipedia_query_run for background info.
STRICT RULES:
- Call each tool AT MOST ONE TIME total.
- Never repeat a tool with a reworded query.
- After tool calls, answer directly with what you have."""
)

# Notes agent
notes_toolkit = [add_note, list_notes, search_notes]
notes_agent = create_agent(
    llm,
    tools=notes_toolkit,
    system_prompt=f"""You are a notes management assistant. Today's date is {today_str}.
You have tools to add, list, and search notes.
Rules:
- Use the most specific tool for the request.
- Call each tool AT MOST ONCE per request.
- Give a clear confirmation after using a tool."""
)

# Tasks agent
tasks_toolkit = [add_task, list_pending_tasks, complete_task]
tasks_agent = create_agent(
    llm,
    tools=tasks_toolkit,
    system_prompt=f"""You are a task management assistant. Today's date is {today_str}.
You have tools to add, list, and complete tasks.
Rules:
- Use the most specific tool for the request.
- Call each tool AT MOST ONCE per request.
- Give a clear confirmation after using a tool."""
)

# Knowledge agent (RAG) — reuses your existing chroma_db
embedding_model = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
vectorstore = Chroma(
    persist_directory="./data/chroma_db",
    embedding_function=embedding_model
)

@tool
def knowledge_base_search(query: str) -> str:
    """Search uploaded documents and answer using only that content."""
    context = vectorstore.similarity_search(query, k=3)
    prompt = f"""You are a knowledge base assistant.
Answer using ONLY the context below. If the answer is not in the context, say:
"I don't have that information in the uploaded documents."

Context:
{context}

Question:
{query}"""
    return llm.invoke(prompt).content

knowledge_toolkit = [knowledge_base_search]
knowledge_agent = create_agent(
    llm,
    tools=knowledge_toolkit,
    system_prompt=f"""You are a knowledge base assistant. Today's date is {today_str}.
Search uploaded documents to answer questions.
Rules:
- Call knowledge_base_search AT MOST ONCE.
- Only answer from the document content."""
)

print("All agents ready")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8760.60it/s]
C:\Users\harih\AppData\Local\Temp\ipykernel_6700\2428624221.py:46: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


All agents ready


In [10]:
def run_coordinator(user_input: str, thread_id: str = "default") -> str:
    # Step 1 — classify intent
    classification = classifier.invoke({"input": user_input})
    intent = classification.intent
    print(f"[Coordinator] Intent: {intent} | Reason: {classification.reasoning}")
    
    # Step 2 — route to the right agent
    if intent == "notes":
        agent = notes_agent
    elif intent == "tasks":
        agent = tasks_agent
    elif intent == "research":
        agent = research_agent
    elif intent == "knowledge":
        agent = knowledge_agent
    else:
        # chat — no agent needed, just LLM directly
        response = llm.invoke(user_input)
        return response.content
    
    # Step 3 — run the selected agent
    events = agent.stream(
        {"messages": [("user", user_input)]},
        config={
            "recursion_limit": 8,
            "configurable": {"thread_id": thread_id}
        },
        stream_mode="values"
    )
    
    final_message = None
    for event in events:
        final_message = event["messages"][-1]
    
    return final_message.content

print("Coordinator ready")

Coordinator ready


In [11]:
test_cases = [
    ("Save a note titled 'Team standup' with content 'Discussed sprint goals'", "thread1"),
    ("What are my pending tasks?", "thread1"),
    ("Search the web for latest LangChain updates", "thread1"),
    ("What does my uploaded document say about LangChain?", "thread1"),
    ("What is 2 + 2?", "thread1"),
]

for query, tid in test_cases:
    print(f"\n{'='*50}")
    print(f"USER: {query}")
    print(f"RESPONSE: {run_coordinator(query, tid)}")


USER: Save a note titled 'Team standup' with content 'Discussed sprint goals'
[Coordinator] Intent: notes | Reason: The user explicitly requests to save a note with a title and content, which falls under the 'notes' intent.
RESPONSE: Your note titled **“Team standup”** has been saved successfully. Let me know if you’d like to view, edit, or add anything else!

USER: What are my pending tasks?
[Coordinator] Intent: tasks | Reason: The user is asking to retrieve a list of pending tasks, which falls under the 'tasks' intent (listing tasks).
RESPONSE: Here are your pending tasks:

1. **Prepare tomorrow's workshop** – due 2026‑08‑09  
2. **Prepare the client presentation** – due 2026‑08‑09  

Let me know if you’d like to add, edit, or complete any tasks!

USER: Search the web for latest LangChain updates
[Coordinator] Intent: research | Reason: The user explicitly asks to search the web for the latest LangChain updates, which is a request for current information from the internet, matching